In [345]:
#project overview
'''This project aims to predict whether a customer is likely to churn based on customer demographics,
usage behavior, support interactions, payment delays, subscription type, contract length, spending,
and recent activity.

Two classification models are evaluated:
    Logistic Regression
    Random Forest

The models are compared using accuracy, precision, recall, F1-score'''

'This project aims to predict whether a customer is likely to churn based on customer demographics,\nusage behavior, support interactions, payment delays, subscription type, contract length, spending,\nand recent activity.\n\nTwo classification models are evaluated:\n    Logistic Regression\n    Random Forest\n\nThe models are compared using accuracy, precision, recall, F1-score'

In [346]:
#import libraries
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

In [347]:
#load Data
training_data = pd.read_csv(r"D:\ML\customer-churn-prediction\data\raw\customer_churn_dataset-training-master.csv")
testing_data = pd.read_csv(r"D:\ML\customer-churn-prediction\data\raw\customer_churn_dataset-testing-master.csv")

In [348]:
#exploratory data analysis
training_data.info()
training_data.describe()
training_data.isnull().sum()
training_data["Churn"].value_counts()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 440833 entries, 0 to 440832
Data columns (total 12 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   CustomerID         440832 non-null  float64
 1   Age                440832 non-null  float64
 2   Gender             440832 non-null  object 
 3   Tenure             440832 non-null  float64
 4   Usage Frequency    440832 non-null  float64
 5   Support Calls      440832 non-null  float64
 6   Payment Delay      440832 non-null  float64
 7   Subscription Type  440832 non-null  object 
 8   Contract Length    440832 non-null  object 
 9   Total Spend        440832 non-null  float64
 10  Last Interaction   440832 non-null  float64
 11  Churn              440832 non-null  float64
dtypes: float64(9), object(3)
memory usage: 40.4+ MB


Churn
1.0    249999
0.0    190833
Name: count, dtype: int64

In [349]:
#data preprocessing
training_data=training_data.dropna()

x=training_data.drop("Churn", axis=1)
y=training_data["Churn"]
x = x.drop(columns=["CustomerID"])

x_test=testing_data.drop("Churn", axis=1)
y_test=testing_data["Churn"]

numerical_features=x.select_dtypes(include=['int64', 'float64']).columns
categorical_features=x.select_dtypes(include=['object']).columns

preprocessor=ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(), categorical_features)
    ]
)

x_processed = preprocessor.fit_transform(x)
x_test_processed = preprocessor.transform(x_test)



In [350]:
#logistic regression model
model=LogisticRegression()
model.fit(x_processed, y)
y_pred=model.predict(x_test_processed)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.571302078478889
Precision: 0.5251756033103832
Recall: 0.9905880038041518
F1: 0.6864297969525844

Confusion Matrix:
[[ 6571 27310]
 [  287 30206]]


In [351]:
#random forest model
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(x_processed, y)

y_proba = rf_model.predict_proba(x_test_processed)[:, 1]
y_pred_08 = (y_proba >= 0.8).astype(int)
print("Accuracy:", accuracy_score(y_test, y_pred_08))
print("Precision:", precision_score(y_test, y_pred_08))
print("Recall:", recall_score(y_test, y_pred_08))
print("F1:", f1_score(y_test, y_pred_08))

print(confusion_matrix(y_test, y_pred_08))

Accuracy: 0.5094292726877311
Precision: 0.49122778701357395
Recall: 0.998097924113731
F1: 0.6584099513250405
[[ 2359 31522]
 [   58 30435]]


In [353]:
#concat the random split
full_data=pd.concat([training_data, testing_data], ignore_index=True)
full_data=full_data.drop(columns=["CustomerID"])
X=full_data.drop(columns=["Churn"])
Y=full_data["Churn"]

x_train, x_test, y_train, y_test=train_test_split(X, Y, test_size=0.2, random_state=42,stratify=Y)

x_train_processed=preprocessor.fit_transform(x_train)
x_test_processed=preprocessor.transform(x_test)

model=LogisticRegression(max_iter=1000)
model.fit(x_train_processed, y_train)

threshold = 0.48

y_pred = (y_proba >= threshold).astype(int)
y_proba=model.predict_proba(x_test_processed)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))

cm=confusion_matrix(y_test, y_pred)
print(cm)
print("----------------------------")
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(x_train_processed, y_train)

rf_pred = rf_model.predict(x_test_processed)
rf_proba = rf_model.predict_proba(x_test_processed)[:, 1]
    
print("Accuracy:", accuracy_score(y_test, rf_pred))
print("Precision:", precision_score(y_test, rf_pred))
print("Recall:", recall_score(y_test, rf_pred))
print("F1:", f1_score(y_test, rf_pred))

Accuracy: 0.8500326596860711
Precision: 0.8677035813067998
Recall: 0.8611918216010981
F1 Score: 0.8644354384175635
[[37577  7366]
 [ 7787 48312]]
----------------------------
Accuracy: 0.935897943429465
Precision: 0.8984038795041428
Recall: 0.9973261555464447
F1: 0.9452840549102429


In [ ]:
#conclusion
'''
the split of the training and testing data in two seperate files presents low performance because
they have different distributions like :
    In the training data:      Monthly contracts → 100% churn
    
    But in the testing data:   Monthly contracts → ~51.6% churn

resulting in 0.57 accuracy for the logistic regression model and 0.51 accuracy 
the random forest model.

when doing a random split of the data, we get better performance with 0.85 accuracy for 
the logistic regression model and 0.93 accuracy for the random forest model.

Overall, the results show that the choice of train-test split can have a significant
impact on model evaluation. Among the models tested, Random Forest achieved the 
strongest overall performance on the newly created stratified split and was therefore 
selected as the final model.

'''